# Mapa de Temperatura Média por Distrito (interativo)

Lê a `gold_monthly` do HDFS e desenha um choropleth da **temperatura média** do mês escolhido.
Usa os dropdowns de **Ano** e **Mês** e carrega em **Mostrar mapa**.

In [23]:
!pip install -q ipywidgets folium
import os, json, unicodedata
import pandas as pd
import folium
import ipywidgets as widgets
from IPython.display import display
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

GOLD_MONTHLY = "hdfs://hadoop-master:39000/2250055/gold/gold_monthly"
GEO_PATH = "hdfs://hadoop-master:39000/2250055/geo/distritos.geojson"


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [24]:
# Cria o GeoJSON dos distritos (formas aproximadas) se ainda nao existir.
os.makedirs(os.path.dirname(GEO_PATH), exist_ok=True)
geojson_distritos = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"name": "Aveiro"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.768, 40.932], [-8.645, 40.93], [-8.529, 40.805], [-8.514, 40.676], [-8.629, 40.574], [-8.768, 40.6], [-8.768, 40.932]]]]}}, {"type": "Feature", "properties": {"name": "Beja"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.285, 38.32], [-7.9, 38.32], [-7.6, 37.9], [-7.9, 37.6], [-8.285, 37.7], [-8.285, 38.32]]]]}}, {"type": "Feature", "properties": {"name": "Braga"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.7, 41.7], [-8.2, 41.7], [-8.2, 41.4], [-8.7, 41.4], [-8.7, 41.7]]]]}}, {"type": "Feature", "properties": {"name": "Bragança"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-7.3, 41.9], [-6.5, 41.9], [-6.5, 41.3], [-7.3, 41.3], [-7.3, 41.9]]]]}}, {"type": "Feature", "properties": {"name": "Castelo Branco"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.1, 40.3], [-7.2, 40.3], [-7.2, 39.6], [-8.1, 39.6], [-8.1, 40.3]]]]}}, {"type": "Feature", "properties": {"name": "Coimbra"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.7, 40.4], [-8.1, 40.4], [-8.1, 39.9], [-8.7, 39.9], [-8.7, 40.4]]]]}}, {"type": "Feature", "properties": {"name": "Évora"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.2, 38.9], [-7.3, 38.9], [-7.3, 38.3], [-8.2, 38.3], [-8.2, 38.9]]]]}}, {"type": "Feature", "properties": {"name": "Faro"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.9, 37.3], [-7.4, 37.3], [-7.4, 36.9], [-8.9, 36.9], [-8.9, 37.3]]]]}}, {"type": "Feature", "properties": {"name": "Guarda"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-7.7, 41.1], [-6.7, 41.1], [-6.7, 40.5], [-7.7, 40.5], [-7.7, 41.1]]]]}}, {"type": "Feature", "properties": {"name": "Leiria"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-9.1, 39.9], [-8.5, 39.9], [-8.5, 39.3], [-9.1, 39.3], [-9.1, 39.9]]]]}}, {"type": "Feature", "properties": {"name": "Lisboa"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-9.5, 39.1], [-8.8, 39.1], [-8.8, 38.6], [-9.5, 38.6], [-9.5, 39.1]]]]}}, {"type": "Feature", "properties": {"name": "Portalegre"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.1, 39.4], [-7.2, 39.4], [-7.2, 38.8], [-8.1, 38.8], [-8.1, 39.4]]]]}}, {"type": "Feature", "properties": {"name": "Porto"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.8, 41.4], [-8.3, 41.4], [-8.3, 41.0], [-8.8, 41.0], [-8.8, 41.4]]]]}}, {"type": "Feature", "properties": {"name": "Santarém"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.9, 39.6], [-8.1, 39.6], [-8.1, 38.9], [-8.9, 38.9], [-8.9, 39.6]]]]}}, {"type": "Feature", "properties": {"name": "Setúbal"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-9.2, 38.6], [-8.4, 38.6], [-8.4, 37.9], [-9.2, 37.9], [-9.2, 38.6]]]]}}, {"type": "Feature", "properties": {"name": "Viana do Castelo"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.9, 41.9], [-8.4, 41.9], [-8.4, 41.6], [-8.9, 41.6], [-8.9, 41.9]]]]}}, {"type": "Feature", "properties": {"name": "Vila Real"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-7.9, 41.6], [-7.2, 41.6], [-7.2, 41.1], [-7.9, 41.1], [-7.9, 41.6]]]]}}, {"type": "Feature", "properties": {"name": "Viseu"}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-8.3, 41.0], [-7.5, 41.0], [-7.5, 40.5], [-8.3, 40.5], [-8.3, 41.0]]]]}}]}
with open(GEO_PATH, 'w') as f:
    json.dump(geojson_distritos, f, ensure_ascii=False)
print('GeoJSON pronto em', GEO_PATH)

GeoJSON pronto em hdfs://hadoop-master:39000/2250055/geo/distritos.geojson


In [25]:
# Le a gold_monthly do HDFS e traz so o que precisamos para pandas (tabela pequena).
df_mes = (spark.read.parquet(GOLD_MONTHLY)
          .select("district", "year_month", "temp_media_mes")
          .toPandas())

def normaliza(s):
    """Tira acentos e poe em maiusculas, para casar distrito <-> geojson."""
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore").decode("utf-8").upper())

df_mes["district"] = df_mes["district"].apply(normaliza)

geo_str = spark.sparkContext.wholeTextFiles(GEO_PATH).collect()[0][1]
geo = json.loads(geo_str)

for feat in geo["features"]:
    feat["properties"]["name"] = normaliza(feat["properties"]["name"])

print("Meses disponiveis:", sorted(df_mes["year_month"].unique()))

Meses disponiveis: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


In [26]:
def gerar_mapa(year_month):
    d = df_mes[df_mes["year_month"] == year_month]
    m = folium.Map(location=[39.5, -8.0], zoom_start=6, width="100%", height="650px")
    if d.empty:
        print("Sem dados para", year_month)
        return m
    folium.Choropleth(
        geo_data=geo,
        data=d,
        columns=["district", "temp_media_mes"],
        key_on="feature.properties.name",
        fill_color="YlOrRd",
        fill_opacity=0.85,
        line_opacity=0.3,
        nan_fill_color="lightgray",
        legend_name=f"Temperatura media (C) - {year_month}",
    ).add_to(m)
    return m

# --- Interface: dropdowns de Ano e Mes (dependentes) + botao ---
meses = sorted(df_mes["year_month"].unique())
anos = sorted({ym[:4] for ym in meses})

dd_ano = widgets.Dropdown(options=anos, description="Ano:")
dd_mes = widgets.Dropdown(description="Mes:")
saida = widgets.Output()

def atualizar_meses(*_):
    dd_mes.options = [ym[5:] for ym in meses if ym.startswith(dd_ano.value)]
atualizar_meses()
dd_ano.observe(atualizar_meses, "value")

def mostrar(*_):
    with saida:
        saida.clear_output(wait=True)
        ym = f"{dd_ano.value}-{dd_mes.value}"
        display(gerar_mapa(ym))

botao = widgets.Button(description="Mostrar mapa", button_style="primary")
botao.on_click(mostrar)

display(widgets.HBox([dd_ano, dd_mes, botao]), saida)

Output()

In [14]:
spark.stop()